In [ ]:
import asyncio
import os
import re
import pandas as pd
import requests
from playwright.async_api import async_playwright

# ВСТАВЬТЕ СЮДА ВАШУ ССЫЛКУ ИЗ GOOGLE APPS SCRIPT (заканчивается на /exec)
WEB_APP_URL = "https://script.google.com/macros/s/AKfycbyYz_Ggn887A6py2nWbm0JbueQUWdF_UpOs3HpwxGd4UdGJ-cPeL1i_nfODbEE2HoGH/exec"

CONCURRENCY = 4

# Если True — при старте очистит старый CSV с нулями, но СОХРАНИТ все 3078 собранных ссылок!
# (После первого запуска поменяйте на False, если будете перезапускать)
RESET_CSV_WITH_ZEROS = True

GRID_COORDS = [
    (lat, lon)
    for lat in [40.4280, 40.4230, 40.4180, 40.4130, 40.4080]
    for lon in [-3.7140, -3.7080, -3.7020, -3.6960, -3.6900]
]

CATEGORIES = [
    "Restaurante",
    "Bar de tapas",
    "Cafetería",
    "Taberna",
    "Bar",
    "Pizzería",
    "Hamburguesería",
    "Sushi",
    "Comida rápida",
    "Pastelería"
]

CSV_FILENAME = "madrid_centro_restaurants.csv"
URLS_BACKUP_FILE = "collected_urls.txt"


def upload_to_google_sheet(df):
    """Отправляет таблицу в Google Sheets через веб-хук Google Apps Script."""
    if not WEB_APP_URL.startswith("http"):
        print("[!] Укажите корректный WEB_APP_URL. Данные сохранены только в CSV.")
        return

    try:
        rows = [df.columns.values.tolist()] + df.fillna("").values.tolist()
        response = requests.post(WEB_APP_URL, json=rows, timeout=45)
        if response.status_code == 200:
            print(f"[✓] Google Таблица обновлена! Записано заведений: {len(df)}")
        else:
            print(f"[!] Статус ответа таблицы {response.status_code}: {response.text[:100]}")
    except Exception as e:
        print(f"[!] Ошибка отправки в Google Таблицу: {e}")


async def accept_cookies(page):
    try:
        btn = page.locator(
            'button:has-text("Принять все"), '
            'button:has-text("Aceptar todo"), '
            'button:has-text("Accept all")'
        )
        if await btn.count() > 0:
            await btn.first.click(timeout=3000)
            await page.wait_for_timeout(1000)
    except Exception:
        pass


async def get_urls_for_task(context, sem, category, lat, lon, idx, total_tasks, all_urls, max_scrolls=25):
    async with sem:
        page = await context.new_page()
        urls = set()
        try:
            search_url = f"https://www.google.com/maps/search/{category}/@{lat},{lon},17z?hl=ru"
            await page.goto(search_url, wait_until="domcontentloaded", timeout=30000)
            await accept_cookies(page)

            feed_selector = 'div[role="feed"]'
            try:
                await page.wait_for_selector(feed_selector, timeout=8000)
            except Exception:
                await page.close()
                return urls

            last_count = 0
            stagnant_rounds = 0

            for _ in range(max_scrolls):
                links = await page.locator('div[role="feed"] a[href*="/maps/place/"]').all()
                for link in links:
                    href = await link.get_attribute("href")
                    if href:
                        clean_url = href.split("?")[0]
                        urls.add(clean_url)

                if len(urls) == last_count:
                    stagnant_rounds += 1
                    if stagnant_rounds >= 3:
                        break
                else:
                    stagnant_rounds = 0
                last_count = len(urls)

                await page.evaluate(f'''
                    const feed = document.querySelector('{feed_selector}');
                    if (feed) feed.scrollTop = feed.scrollHeight;
                ''')
                await page.wait_for_timeout(1400)

            new_found = urls - all_urls
            all_urls.update(urls)
            with open(URLS_BACKUP_FILE, "w", encoding="utf-8") as f:
                for u in sorted(all_urls):
                    f.write(u + "\n")

            print(f"[Поиск {idx}/{total_tasks}] {category} @ ({lat}, {lon}) -> +{len(new_found)} новых (Всего: {len(all_urls)})")
        except Exception as e:
            print(f"[!] Ошибка поиска {category} @ ({lat}, {lon}): {e}")
        finally:
            await page.close()

        return urls


async def parse_restaurant_ru(context, sem, url, idx, total):
    """Гарантированно извлекает название, тип, чек, адрес и точное распределение звезд."""
    async with sem:
        page = await context.new_page()
        data = {
            "Название": "",
            "Тип заведения": "",
            "Средний чек": "Нет данных",
            "Адрес": "",
            "Средняя оценка": "",
            "Всего отзывов": 0,
            "5 звезд": 0,
            "4 звезды": 0,
            "3 звезды": 0,
            "2 звезды": 0,
            "1 звезда": 0,
            "Ссылка": url
        }

        star_map = {
            "5": "5 звезд",
            "4": "4 звезды",
            "3": "3 звезды",
            "2": "2 звезды",
            "1": "1 звезда"
        }

        try:
            # Переходим на карточку с русской локалью
            await page.goto(url + "?hl=ru", wait_until="domcontentloaded", timeout=35000)

            # Ждем появления основного заголовка карточки (названия)
            try:
                await page.wait_for_selector('h1.DUwDvf', timeout=10000)
            except Exception:
                pass
            await page.wait_for_timeout(2000)

            # 1. Название ресторана
            try:
                name_el = page.locator('h1.DUwDvf')
                if await name_el.count() > 0:
                    data["Название"] = (await name_el.first.inner_text()).strip()
            except Exception:
                pass

            # 2. Тип заведения (например, "Ресторан русской кухни")
            try:
                cat_btn = page.locator('button.DkEaL, button[jsaction*="category"]')
                if await cat_btn.count() > 0:
                    data["Тип заведения"] = (await cat_btn.first.inner_text()).strip()
            except Exception:
                pass

            # 3. Средняя оценка и общее количество отзывов
            try:
                rating_box = page.locator('div.F7nice')
                if await rating_box.count() > 0:
                    text = await rating_box.first.inner_text()
                    lines = [line.strip() for line in text.split('\n') if line.strip()]
                    if len(lines) >= 1:
                        data["Средняя оценка"] = lines[0].replace('.', ',')
                    if len(lines) >= 2:
                        total_clean = re.sub(r'[^\d]', '', lines[1])
                        data["Всего отзывов"] = int(total_clean) if total_clean.isdigit() else 0
            except Exception:
                pass

            try:
                await page.wait_for_function(
                    r"""() => /\d\s*[–\-—]\s*\d+\s*€|€\s*\d|€{1,4}/.test(
                        document.querySelector('div.lMbq3e')?.innerText || '')""",
                    timeout=4000
                )
            except Exception:
                pass

            # 4. Средний чек (сбор текста из верхней панели шапки)
            try:
                raw_candidates = await page.evaluate('''() => {
                    const out = [];
                    // a) шапка карточки (название + оценка + цена + тип)
                    const h1 = document.querySelector('h1.DUwDvf');
                    if (h1) {
                        let box = h1.closest('div.lMbq3e') || h1.parentElement?.parentElement;
                        if (box) out.push(box.innerText || '');
                    }
                    // b) элементы, у которых в aria-label есть цена
                    document.querySelectorAll('[aria-label]').forEach(el => {
                        const l = el.getAttribute('aria-label') || '';
                        if (/€|Цена|Price|Precio/i.test(l) && l.length < 80) out.push(l);
                    });
                    // c) начало текста основной панели
                    const main = document.querySelector('div[role="main"]');
                    if (main) out.push((main.innerText || '').slice(0, 700));
                    return out;
                }''')

                price_regex = re.compile(
                    r'€\s*(\d+(?:[.,]\d+)?)\s*[–\-—]\s*(\d+(?:[.,]\d+)?)'          # 1,2: €10–20
                    r'|(\d+(?:[.,]\d+)?)\s*[–\-—]\s*(\d+(?:[.,]\d+)?)\s*€'          # 3,4: 10–20 €
                    r'|(?:Более|Свыше|От|До|Más de|More than)\s*(\d+)\s*€'          # 5:   Более 50 €
                    r'|(\d+)\s*€\s*\+'                                              # 6:   50 €+
                    r'|€\s*(\d+)\s*\+'                                              # 7:   €50+
                    r'|(?<![\w€])(€{1,4})(?![\w€])',                                # 8:   €€
                    re.IGNORECASE
                )

                found_price = ""
                fallback_symbols = ""
                for cand in raw_candidates:
                    txt = re.sub(r'[\s\u202f\xa0\u2009]+', ' ', cand)
                    for m in price_regex.finditer(txt):
                        g = m.groups()
                        if g[0] and g[1]:
                            found_price = f"{g[0]}–{g[1]}"
                        elif g[2] and g[3]:
                            found_price = f"{g[2]}–{g[3]}"
                        elif g[4]:
                            found_price = f"{g[4]}+"
                        elif g[5]:
                            found_price = f"{g[5]}+"
                        elif g[6]:
                            found_price = f"{g[6]}+"
                        elif g[7] and not fallback_symbols:
                            fallback_symbols = g[7]  # только символы €€ — запасной вариант
                        if found_price:
                            break
                    if found_price:
                        break

                if found_price:
                    data["Средний чек"] = found_price
                elif fallback_symbols:
                    data["Средний чек"] = fallback_symbols
            except Exception:
                pass

            # 5. Адрес
            try:
                addr_btn = page.locator('button[data-item-id="address"]')
                if await addr_btn.count() > 0:
                    raw_addr = await addr_btn.first.get_attribute("aria-label")
                    if raw_addr:
                        data["Адрес"] = re.sub(
                            r'^(Адрес|Dirección|Address):\s*', '', raw_addr, flags=re.IGNORECASE
                        ).strip()
                    else:
                        data["Адрес"] = (await addr_btn.first.inner_text()).strip()
            except Exception:
                pass

            # 6. Количество разных оценок (5★, 4★, 3★, 2★, 1★) через принудительный клик на отзывы
            try:
                reviews_tab = page.locator(
                    'button[role="tab"]:has-text("Отзывы"), '
                    'button[role="tab"]:has-text("Reseñas"), '
                    'button[role="tab"]:has-text("Reviews")'
                )
                if await reviews_tab.count() > 0:
                    await reviews_tab.first.click()
                    # Даем время на отрисовку гистограммы
                    try:
                        await page.wait_for_selector('tr.BHOKXe', timeout=5000)
                    except Exception:
                        pass
                    await page.wait_for_timeout(1000)

                # Универсальный сбор строк гистограммы из DOM с помощью JS
                stars_data = await page.evaluate('''() => {
                    const rows = Array.from(document.querySelectorAll('tr.BHOKXe'));
                    return rows.map(r => {
                        let label = r.getAttribute('aria-label') || '';
                        let text = r.innerText || '';
                        return { label, text };
                    });
                }''')

                for idx_row, row in enumerate(stars_data):
                    combined_str = f"{row['label']} {row['text']}".replace('\xa0', ' ').replace('\u202f', ' ')
                    # Извлекаем все числа из строки гистограммы
                    numbers = re.findall(r'\d[\d\s\.,]*', combined_str)
                    clean_numbers = [re.sub(r'[^\d]', '', n) for n in numbers if re.sub(r'[^\d]', '', n)]

                    if len(clean_numbers) >= 2 and clean_numbers[0] in star_map:
                        star_num = clean_numbers[0]
                        data[star_map[star_num]] = int(clean_numbers[1])
                    elif len(clean_numbers) == 1 and len(stars_data) == 5:
                        # Если в строке только количество отзывов, распределение идет сверху вниз (5 -> 1)
                        star_num = str(5 - idx_row)
                        data[star_map[star_num]] = int(clean_numbers[0])
            except Exception:
                pass

            print(
                f"[{idx}/{total}] {data['Название']} | Чек: {data['Средний чек']} | "
                f"Отзывов: {data['Всего отзывов']} (5★:{data['5 звезд']}, 4★:{data['4 звезды']}, "
                f"3★:{data['3 звезды']}, 2★:{data['2 звезды']}, 1★:{data['1 звезда']})"
            )
        except Exception as e:
            print(f"[{idx}/{total}] Ошибка на карточке: {e}")
        finally:
            await page.close()

        return data


async def main():
    sem = asyncio.Semaphore(CONCURRENCY)
    all_urls = set()

    if RESET_CSV_WITH_ZEROS and os.path.exists(CSV_FILENAME):
        os.remove(CSV_FILENAME)
        print("[!] Старый CSV с нулями удален. Начинаем чистый сбор карточек по готовой базе ссылок!")

    if os.path.exists(URLS_BACKUP_FILE):
        with open(URLS_BACKUP_FILE, "r", encoding="utf-8") as f:
            all_urls = set(line.strip() for line in f if line.strip())
        print(f"[=] В кэше уже есть {len(all_urls)} уникальных ссылок.")

    results = []
    parsed_urls = set()
    if os.path.exists(CSV_FILENAME):
        try:
            existing_df = pd.read_csv(CSV_FILENAME)
            results = existing_df.to_dict("records")
            parsed_urls = set(existing_df["Ссылка"].dropna().tolist())
            print(f"[=] В CSV уже спарсено {len(parsed_urls)} заведений.")
        except Exception:
            pass

    async with async_playwright() as p:
        browser = await p.chromium.launch(
            headless=True,
            args=["--no-sandbox", "--disable-dev-shm-usage"]
        )
        context = await browser.new_context(
            locale="ru-RU",
            viewport={"width": 1366, "height": 800},
            user_agent="Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36"
        )

        init_page = await context.new_page()
        await init_page.goto("https://www.google.com/maps?hl=ru", wait_until="domcontentloaded")
        await accept_cookies(init_page)
        await init_page.close()

        # Если у нас в кэше уже больше 2500 ссылок — сразу начинаем парсить их и отправлять в таблицу!
        if len(all_urls) > 0:
            initial_to_parse = [u for u in sorted(all_urls) if u not in parsed_urls]
            print(f"\n[->] Сразу запускаем парсинг {len(initial_to_parse)} уже найденных в кэше заведений...")
            batch_size = 20
            for start_idx in range(0, len(initial_to_parse), batch_size):
                batch_urls = initial_to_parse[start_idx:start_idx + batch_size]
                batch_tasks = [
                    parse_restaurant_ru(context, sem, url, start_idx + j + 1, len(initial_to_parse))
                    for j, url in enumerate(batch_urls)
                ]
                batch_data = await asyncio.gather(*batch_tasks)
                results.extend(batch_data)
                parsed_urls.update(batch_urls)

                df_temp = pd.DataFrame(results)
                df_temp.to_csv(CSV_FILENAME, index=False, encoding="utf-8-sig")
                upload_to_google_sheet(df_temp)

        # Добираем оставшиеся волны, если нужно
        for wave_idx, category in enumerate(CATEGORIES, 1):
            print(f"\n================ ВОЛНА {wave_idx}/{len(CATEGORIES)}: Категория '{category}' ================")
            search_tasks = [
                get_urls_for_task(context, sem, category, lat, lon, i, len(GRID_COORDS), all_urls)
                for i, (lat, lon) in enumerate(GRID_COORDS, 1)
            ]
            await asyncio.gather(*search_tasks)

            new_urls_to_parse = [u for u in sorted(all_urls) if u not in parsed_urls]
            if not new_urls_to_parse:
                continue

            print(f"\n[->] Новых карточек для парсинга: {len(new_urls_to_parse)}")
            batch_size = 20
            for start_idx in range(0, len(new_urls_to_parse), batch_size):
                batch_urls = new_urls_to_parse[start_idx:start_idx + batch_size]
                batch_tasks = [
                    parse_restaurant_ru(context, sem, url, start_idx + j + 1, len(new_urls_to_parse))
                    for j, url in enumerate(batch_urls)
                ]
                batch_data = await asyncio.gather(*batch_tasks)
                results.extend(batch_data)
                parsed_urls.update(batch_urls)

                df_temp = pd.DataFrame(results)
                df_temp.to_csv(CSV_FILENAME, index=False, encoding="utf-8-sig")
                upload_to_google_sheet(df_temp)

        await browser.close()

    print(f"\n[✓] ПОЛНЫЙ ЦИКЛ ЗАВЕРШЕН! Всего уникальных заведений в вашей таблице: {len(results)}")


await main()

[!] Старый CSV с нулями удален. Начинаем чистый сбор карточек по готовой базе ссылок!
[=] В кэше уже есть 3149 уникальных ссылок.

[->] Сразу запускаем парсинг 3149 уже найденных в кэше заведений...
[1/3149] ] La Prensa. Bar Restaurante en Gran Vía | Чек: Нет данных | Отзывов: 813 (5★:508, 4★:136, 3★:74, 2★:24, 1★:71)
[4/3149] Ástor gastro-place | Чек: Нет данных | Отзывов: 2835 (5★:2589, 4★:188, 3★:35, 2★:11, 1★:12)
[3/3149] Ástor "Puerta Cerrada" | Чек: Нет данных | Отзывов: 1008 (5★:924, 4★:48, 3★:22, 2★:8, 1★:6)
[2/3149] Álvarez Selección | Чек: Нет данных | Отзывов: 195 (5★:171, 4★:14, 3★:1, 2★:0, 1★:9)
[8/3149] Ópera Victoria | Чек: Нет данных | Отзывов: 4775 (5★:2927, 4★:1055, 3★:356, 2★:189, 1★:248)
[5/3149] Ático Restaurante & Terraza | Чек: Нет данных | Отзывов: 1705 (5★:1186, 4★:314, 3★:101, 2★:49, 1★:55)
[7/3149] Ñam Burger | Чек: Нет данных | Отзывов: 186 (5★:172, 4★:8, 3★:1, 2★:1, 1★:4)
[6/3149] Éccolo | Restaurante Italiano & Pizzería | Чек: Нет данных | Отзывов: 1909 (5